# COMM117 Workshop (Lecture 2) — **Solution Notebook**
**Topic:** Data Preprocessing, Quality Filtering, and Analysis  
**Dataset:** `tweet_eval` → `sentiment`

This notebook provides one reasonable reference implementation. Thresholds and design choices can be varied.


In [3]:
# Install dependencies (run once)
!pip install datasets langdetect transformers

  Using cached langdetect-1.0.9.tar.gz (981 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=fcfd7568d5b3978b7bfe87e6e886241fb9a241c202aa3f4ae23a4a02a060b8b1
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [4]:
import re
import html
import random
import hashlib
from collections import Counter
from typing import List, Dict, Tuple

import numpy as np
from datasets import load_dataset

from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

DetectorFactory.seed = 42

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [5]:
# Load dataset
ds = load_dataset("tweet_eval", "sentiment")
ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

## 1) Preprocessing utilities

We implement:
- normalization (HTML unescape, remove control chars, collapse whitespace)
- remove URLs and @mentions
- keep hashtag words but remove '#'
- optional PII redaction (emails/phones)


In [6]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
WHITESPACE_RE = re.compile(r"\s+")
NONPRINT_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
PHONE_RE = re.compile(r"\b(\+?\d[\d\s\-]{7,}\d)\b")

def normalize_text(text: str) -> str:
    """Normalize formatting noise without changing semantics."""
    x = html.unescape(text)                 # &amp; &nbsp; etc.
    x = NONPRINT_RE.sub(" ", x)            # remove control chars
    x = x.replace("\u00a0", " ")         # non-breaking space
    x = WHITESPACE_RE.sub(" ", x).strip()  # collapse whitespace
    return x

def redact_pii(text: str) -> str:
    """Basic PII redaction (demo-level)."""
    x = EMAIL_RE.sub("[REDACTED_EMAIL]", text)
    x = PHONE_RE.sub("[REDACTED_PHONE]", x)
    return x

def preprocess(text: str) -> str:
    """End-to-end preprocessing for short, noisy social text."""
    x = normalize_text(text)
    x = URL_RE.sub(" ", x)
    x = MENTION_RE.sub(" ", x)
    x = HASHTAG_RE.sub(r"\1", x)          # keep hashtag content
    x = redact_pii(x)
    x = WHITESPACE_RE.sub(" ", x).strip()
    return x


In [7]:
# Sanity check: show a few examples
for i in range(8):
    raw = ds["train"][i]["text"]
    print("RAW:  ", raw)
    print("CLEAN:", preprocess(raw))
    print("-"*80)


RAW:   "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"
CLEAN: "QT In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. HappyBirthdayRemusLupin"
--------------------------------------------------------------------------------
RAW:   "Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"
CLEAN: "Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis NHL SJ"
--------------------------------------------------------------------------------
RAW:   Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.
CLEAN: Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.
--------------------------------------------------------------------------------
RAW:   Chase Headley's RBI double in the 8th inning off David Pr

## 2) Language filtering

Two-stage (cascade) example:
1) cheap character heuristic  
2) `langdetect` for confirmation (skip for very short texts)


In [8]:
ALPHA_RE = re.compile(r"[A-Za-z]")
def is_english_heuristic(text: str, min_chars: int = 20, min_alpha_ratio: float = 0.55) -> bool:
    if len(text) < min_chars:
        return False
    letters = ALPHA_RE.findall(text)
    if not letters:
        return False
    return (len(letters) / len(text)) >= min_alpha_ratio

def is_english_langid(text: str, min_chars: int = 20) -> bool:
    if len(text) < min_chars:
        return False
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False

def is_english(text: str) -> bool:
    # cascade: heuristic first, then langid
    if not is_english_heuristic(text):
        return False
    return is_english_langid(text)


## 3) Quality filtering rules

Example rules (tune thresholds as needed):
- length bounds
- language filter
- repetition/spam filter
- punctuation density filter


In [9]:
PUNCT_RE = re.compile(r"[^\w\s]")

def repetition_ratio(text: str) -> float:
    toks = text.lower().split()
    if not toks:
        return 1.0
    c = Counter(toks)
    return max(c.values()) / len(toks)

def keep(clean_text: str) -> bool:
    # Rule 1: length bounds
    if len(clean_text) < 25 or len(clean_text) > 280:
        return False

    # Rule 2: English only
    if not is_english(clean_text):
        return False

    # Rule 3: repetition ratio (spammy text)
    if repetition_ratio(clean_text) > 0.35:
        return False

    # Rule 4: punctuation density
    punct_ratio = len(PUNCT_RE.findall(clean_text)) / max(1, len(clean_text))
    if punct_ratio > 0.25:
        return False

    return True


## 4) Deterministic sampling + pipeline application

We sample `n` items deterministically using a random seed so results are reproducible.


In [10]:
def sample_texts(split: str = "train", n: int = 1500, seed: int = RANDOM_SEED) -> List[str]:
    """Return a deterministic sample of raw texts."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(ds[split]), size=n, replace=False)
    return [ds[split][int(i)]["text"] for i in idx]

raw_texts = sample_texts(n=1500)

clean_texts = [preprocess(t) for t in raw_texts]
kept_mask = [keep(t) for t in clean_texts]

kept_raw = [r for r, k in zip(raw_texts, kept_mask) if k]
kept_clean = [c for c, k in zip(clean_texts, kept_mask) if k]

print("Total sampled:", len(raw_texts))
print("Kept:", len(kept_clean), f"({len(kept_clean)/len(raw_texts):.1%})")


Total sampled: 1500
Kept: 1490 (99.3%)


## 5) Before/After metrics

We report:
- retained ratio
- character length statistics
- exact duplicate rate


In [11]:
def char_len_stats(texts: List[str]) -> Dict[str, float]:
    lens = np.array([len(t) for t in texts], dtype=np.int64)
    return {
        "mean": float(lens.mean()) if len(lens) else 0.0,
        "min": int(lens.min()) if len(lens) else 0,
        "max": int(lens.max()) if len(lens) else 0,
    }

def exact_duplicate_rate(texts: List[str]) -> float:
    if not texts:
        return 0.0
    uniq = len(set(texts))
    return 1.0 - (uniq / len(texts))

print("RAW char length stats:", char_len_stats(raw_texts))
print("KEPT char length stats:", char_len_stats(kept_clean))
print("RAW exact dup rate:", exact_duplicate_rate(raw_texts))
print("KEPT exact dup rate:", exact_duplicate_rate(kept_clean))


RAW char length stats: {'mean': 106.34066666666666, 'min': 31, 'max': 169}
KEPT char length stats: {'mean': 103.41409395973155, 'min': 28, 'max': 169}
RAW exact dup rate: 0.0
KEPT exact dup rate: 0.0


## 6) Qualitative inspection (RAW → CLEANED)

Print a small set of examples that show typical noise removal and
drop decisions.


In [13]:
from typing import List

def show_drop_examples(raw_list: List[str], clean_list: List[str], keep_list: List[bool], k: int = 12):
    shown = 0
    for raw, clean, keep_flag in zip(raw_list, clean_list, keep_list):
        if keep_flag:
            continue  # skip KEEP

        print("RAW:  ", raw)
        print("CLEAN:", clean)
        print("DECISION: DROP")
        print("-"*90)

        shown += 1
        if shown >= k:
            break

show_drop_examples(raw_texts, clean_texts, kept_mask, k=12)


RAW:   @user What if Gangnam Style brought the sun for brazil  kkkkkkkkkkkkkkkkkkkkk very hot here I LOVE U <33
CLEAN: What if Gangnam Style brought the sun for brazil kkkkkkkkkkkkkkkkkkkkk very hot here I LOVE U <33
DECISION: DROP
------------------------------------------------------------------------------------------
RAW:   I watched Sarah G Live again\u002c I noticed @ d end na may disiplina talaga ang mga @user nag-aantayan sila ng turn to go to SG. luv it
CLEAN: I watched Sarah G Live again\u002c I noticed @ d end na may disiplina talaga ang mga nag-aantayan sila ng turn to go to SG. luv it
DECISION: DROP
------------------------------------------------------------------------------------------
RAW:   Eid 3rd day.... Good after none
CLEAN: Eid 3rd day.... Good after none
DECISION: DROP
------------------------------------------------------------------------------------------
RAW:   Good morning! It's Monday........again....:-)...Happy day after National Ice Cream Day...:-)
CLEAN